# AI Programming — Lecture 19
## Lab 4-2: Vision Transformer Transfer Learning on CIFAR-10

이번 실습에서는 **ImageNet-21K로 사전학습된 Vision Transformer(ViT)**를
CIFAR-10 이미지 분류에 fine-tuning합니다.

### 핵심 흐름
```text
CIFAR-10 image (32×32)
→ Resize / ViT preprocessing
→ Pretrained ViT-Base / Patch16 / 224
→ 10-class classification head
→ Fine-tuning
```

### 학습 목표
- Transformer가 이미지에도 적용될 수 있음을 확인합니다.
- Hugging Face에서 pretrained ViT를 불러올 수 있습니다.
- Pretrained model의 classification head를 새로운 task에 맞게 변경할 수 있습니다.
- 작은 subset을 이용해 transfer learning 과정을 짧게 실습합니다.

### Colab 안내
- 인터넷 연결이 필요합니다.
- 처음 실행할 때 TensorFlow와 Hugging Face model weight를 다운로드합니다.
- 실습 시간을 줄이기 위해 **train 1,000개 / validation 200개 / 3 epochs**만 사용합니다.
- GPU runtime을 권장합니다.
- 이 노트북은 TensorFlow용 ViT를 위해 `transformers==4.57.1`과 `tf-keras`를 사용합니다.
- TensorFlow용 모델 로딩 시 알려진 safetensors 호환 문제를 피하기 위해 기존 `tf_model.h5` 가중치를 사용합니다.


In [ ]:
# TensorFlow용 Hugging Face ViT 호환 버전 설치
%pip install -q "tensorflow==2.21.0" "tf-keras==2.21.0" "transformers==4.57.1" "datasets>=4.0,<5.0" "fsspec"

print("설치 완료. 처음 설치한 경우 import 오류가 계속되면 Runtime → Restart session 후 이 셀부터 다시 실행하세요.")


## 1. Pretrained ViT Fine-Tuning

아래 코드는 다음 과정을 한 번에 수행합니다.

1. CIFAR-10 dataset 로드
2. ViT image processor 적용
3. TensorFlow dataset 구성
4. `google/vit-base-patch16-224-in21k` 로드
5. Output class 수를 10으로 변경
6. 3 epochs fine-tuning
7. Training / validation accuracy 확인

> 코드의 세부 API보다 **pretrained Transformer를 새로운 task에 재사용하는 흐름**을 이해하는 것이 목적입니다.

In [ ]:
# Hugging Face Transformers 4.x의 TensorFlow 모델이 legacy TF-Keras를 사용하도록 설정
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from transformers import TFViTForImageClassification, ViTImageProcessor
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("Note: TensorFlow deprecation warning from Transformers 4.x can be ignored for this lab.")

# 1. 데이터셋 로드
dataset = load_dataset("cifar10")

# 2. Image processor
checkpoint = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(checkpoint)

# 3. 전처리 함수 정의
def transform(example):
    inputs = processor(images=example["img"], return_tensors="np")
    inputs["label"] = example["label"]
    return inputs

# 4. 소규모 샘플 데이터 변환 (속도를 위해 일부만 사용)
train_data = dataset["train"].select(range(1000)).map(transform)
val_data = dataset["test"].select(range(200)).map(transform)

# 5. TensorFlow Dataset 변환
def to_tf_dataset(data):
    pixel_values = np.stack([x["pixel_values"][0] for x in data])
    labels = np.asarray([x["label"] for x in data], dtype=np.int32)

    return (
        tf.data.Dataset.from_tensor_slices(
            ({"pixel_values": pixel_values}, labels)
        )
        .shuffle(len(labels), seed=42)
        .batch(16)
        .prefetch(tf.data.AUTOTUNE)
    )

train_tfds = to_tf_dataset(train_data)
val_tfds = to_tf_dataset(val_data)

# 6. Pretrained ViT 모델 로드
# ImageNet-21K의 기존 classification head와 CIFAR-10의 10-class head 크기가 다르므로
# ignore_mismatched_sizes=True로 새 classification head를 초기화합니다.
model = TFViTForImageClassification.from_pretrained(
    checkpoint,
    num_labels=10,
    ignore_mismatched_sizes=True,
    use_safetensors=False,  # TF용 tf_model.h5를 사용하여 safetensors 변환 오류 회피
)

# 7. 컴파일 및 fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

history = model.fit(
    train_tfds,
    validation_data=val_tfds,
    epochs=3,
)

# 8. 정확도 시각화
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ViT (Hugging Face) on CIFAR-10")
plt.legend()
plt.grid(True)
plt.show()


## 2. 확인할 내용

1. CIFAR-10의 원래 이미지는 `32×32`이지만 ViT 입력은 `224×224`로 처리됩니다.
2. ViT-Base는 이미 대규모 ImageNet-21K에서 representation을 학습한 상태입니다.
3. 새로운 task에서는 기존 classification head 대신 **10-class head**를 사용합니다.
4. `ignore_mismatched_sizes=True`는 pretrained backbone은 유지하면서 class 수가 다른 output head를 새로 초기화하기 위해 사용합니다.
5. `use_safetensors=False`는 PyTorch safetensors 변환 대신 체크포인트의 TensorFlow `tf_model.h5` 가중치를 직접 사용하기 위한 호환 설정입니다.
6. 작은 subset만 사용하므로 높은 CIFAR-10 성능을 목표로 하는 실험은 아닙니다.

### 생각해 보기
- 처음부터 ViT를 학습하는 것과 pretrained ViT를 fine-tuning하는 것은 어떤 차이가 있을까요?
- 작은 dataset에서 transfer learning이 특히 유리한 이유는 무엇일까요?
